In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/dataguruji/candidates-data/candidates.jsonl
/kaggle/input/datasets/dataguruji/candidate-job-data/jd_hybrid_profile.pkl
/kaggle/input/datasets/dataguruji/candidate-job-data/jd_profile.pkl
/kaggle/input/datasets/dataguruji/candidate-job-data/jd_semantic_profile.pkl


In [2]:
try:
    import yake
except:
    !pip install -q yake

try:
    import rank_bm25
except:
    !pip install -q rank-bm25

try:
    import faiss
except:
    !pip install -q faiss-cpu

try:
    import pyarrow
except:
    !pip install -q pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.4/91.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.5/360.5 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 84.0 MB/s eta 0:00:00:00:0100:01


In [3]:
import os
import gzip
import json
import random
import pickle
import warnings

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

import spacy

from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")

In [4]:
class CFG:

    RANDOM_STATE = 42

    MODEL_NAME = "BAAI/bge-base-en-v1.5"

    SPACY_MODEL = "en_core_web_sm"

    OUTPUT_DIR = "/kaggle/working"

CFG = CFG()

In [5]:
nlp = spacy.load(CFG.SPACY_MODEL)

embedding_model = SentenceTransformer(
    CFG.MODEL_NAME
)

print("Models Loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Models Loaded


In [6]:
with open(
    "/kaggle/input/datasets/dataguruji/candidate-job-data/jd_hybrid_profile.pkl",
    "rb"
) as f:

    jd_profile = pickle.load(f)

print(jd_profile.keys())

dict_keys(['raw_text', 'sentences', 'keywords', 'important_sentences', 'noun_chunks', 'entities', 'clusters', 'cluster_summary', 'cluster_embeddings', 'bm25'])


In [7]:
CANDIDATE_PATH = "/kaggle/input/datasets/dataguruji/candidates-data/candidates.jsonl"

candidates = []

with open(
    CANDIDATE_PATH,
    "r",
    encoding="utf-8"
) as f:

    for line in tqdm(f):

        if line.strip():

            candidates.append(
                json.loads(line)
            )

print()

print("="*70)

print("Candidates Loaded")

print("="*70)

print(len(candidates))

0it [00:00, ?it/s]


Candidates Loaded
100000


In [8]:
candidate = candidates[0]

print(candidate.keys())

print()

print(candidate["profile"].keys())

print()

print(candidate["redrob_signals"].keys())

dict_keys(['candidate_id', 'profile', 'career_history', 'education', 'skills', 'certifications', 'languages', 'redrob_signals'])

dict_keys(['anonymized_name', 'headline', 'summary', 'location', 'country', 'years_of_experience', 'current_title', 'current_company', 'current_company_size', 'current_industry'])

dict_keys(['profile_completeness_score', 'signup_date', 'last_active_date', 'open_to_work_flag', 'profile_views_received_30d', 'applications_submitted_30d', 'recruiter_response_rate', 'avg_response_time_hours', 'skill_assessment_scores', 'connection_count', 'endorsements_received', 'notice_period_days', 'expected_salary_range_inr_lpa', 'preferred_work_mode', 'willing_to_relocate', 'github_activity_score', 'search_appearance_30d', 'saved_by_recruiters_30d', 'interview_completion_rate', 'offer_acceptance_rate', 'verified_email', 'verified_phone', 'linkedin_connected'])


In [9]:
print("="*60)

print("Candidate Dataset")

print("="*60)

print("Total Candidates :", len(candidates))

print()

print("Average Skills :",
      np.mean([len(c["skills"]) for c in candidates]))

print()

print("Average Jobs :",
      np.mean([len(c["career_history"]) for c in candidates]))

print()

print("Average Education :",
      np.mean([len(c["education"]) for c in candidates]))

Candidate Dataset
Total Candidates : 100000

Average Skills : 9.60302

Average Jobs : 3.00171

Average Education : 1.39778


In [10]:
# ==========================================================
# Candidate Semantic Builder V2
# Natural Language Optimized
# ==========================================================

class CandidateSemanticBuilder:

    def __init__(self, candidate):

        self.candidate = candidate

    #########################################################
    # Profile View
    #########################################################

    def profile_document(self):

        p = self.candidate["profile"]

        text = f"""
{p.get('headline','')}

{p.get('summary','')}

The candidate currently works as
{p.get('current_title','')}

at
{p.get('current_company','')}

in the
{p.get('current_industry','')}
industry.

The candidate has
{p.get('years_of_experience',0)}
years of professional experience.

Location:
{p.get('location','')},
{p.get('country','')}.
"""

        return " ".join(text.split())

    #########################################################
    # Career View
    #########################################################

    def career_document(self):

        docs = []

        for job in self.candidate["career_history"]:

            current = "currently works" if job["is_current"] else "worked"

            docs.append(

f"""
The candidate {current}
as a

{job['title']}

at

{job['company']}

for

{job['duration_months']} months.

Industry:

{job['industry']}.

Responsibilities:

{job['description']}
"""
            )

        return "\n".join(docs)

    #########################################################
    # Skills View
    #########################################################

    def skills_document(self):

        docs = []

        for skill in self.candidate["skills"]:

            docs.append(

f"""
The candidate has

{skill['proficiency']}

proficiency in

{skill['name']}

with

{skill.get('duration_months',0)}

months of experience
and

{skill['endorsements']}

endorsements.
"""
            )

        return "\n".join(docs)

    #########################################################
    # Education View
    #########################################################

    def education_document(self):

        docs = []

        for edu in self.candidate["education"]:

            docs.append(

f"""
Completed

{edu['degree']}

in

{edu['field_of_study']}

from

{edu['institution']}.

Institution Tier:

{edu.get('tier','unknown')}.

Academic Grade:

{edu.get('grade','Not Available')}.
"""
            )

        return "\n".join(docs)

    #########################################################
    # Certifications
    #########################################################

    def certification_document(self):

        certs = self.candidate.get("certifications",[])

        if len(certs)==0:

            return ""

        docs=[]

        for cert in certs:

            docs.append(

f"""
Certified in

{cert['name']}

issued by

{cert['issuer']}

during

{cert['year']}.
"""
            )

        return "\n".join(docs)

In [11]:
# ==========================================================
# Build Semantic Documents for ALL Candidates
# ==========================================================

from tqdm.auto import tqdm

candidate_documents = []

for candidate in tqdm(candidates, desc="Building Candidate Documents"):

    builder = CandidateSemanticBuilder(candidate)

    candidate_documents.append({

        "candidate_id": candidate["candidate_id"],

        "profile_document": builder.profile_document(),

        "career_document": builder.career_document(),

        "skills_document": builder.skills_document(),

        "education_document": builder.education_document(),

        "certification_document": builder.certification_document()

    })

print()

print("="*60)
print("Finished")
print("="*60)

print("Total Candidates :", len(candidate_documents))

Building Candidate Documents:   0%|          | 0/100000 [00:00<?, ?it/s]


Finished
Total Candidates : 100000


In [12]:
candidate_documents_df = pd.DataFrame(candidate_documents)

print(candidate_documents_df.shape)

candidate_documents_df.head()

(100000, 6)


,candidate_id,profile_document,career_document,skills_document,education_document,certification_document
0,CAND_0000001,"Backend Engineer | SQL, Spark, Cloud Software ...",\nThe candidate currently works\nas a\n\nBacke...,\nThe candidate has\n\nintermediate\n\nprofici...,\nCompleted\n\nB.E.\n\nin\n\nComputer Science\...,
1,CAND_0000002,Operations Manager | 12.5+ yrs experience Prof...,\nThe candidate currently works\nas a\n\nOpera...,\nThe candidate has\n\nintermediate\n\nprofici...,\nCompleted\n\nB.Sc\n\nin\n\nMathematics\n\nfr...,
2,CAND_0000003,Customer Support | 1.1+ yrs experience Profess...,\nThe candidate currently works\nas a\n\nCusto...,\nThe candidate has\n\nintermediate\n\nprofici...,\nCompleted\n\nM.E.\n\nin\n\nChemical Engineer...,
3,CAND_0000004,Marketing Manager | Driving business outcomes ...,\nThe candidate currently works\nas a\n\nMarke...,\nThe candidate has\n\nintermediate\n\nprofici...,\nCompleted\n\nB.Tech\n\nin\n\nMachine Learnin...,\nCertified in\n\nAWS Certified Cloud Practiti...
4,CAND_0000005,Accountant | Helping teams scale Professional ...,\nThe candidate currently works\nas a\n\nAccou...,\nThe candidate has\n\nbeginner\n\nproficiency...,\nCompleted\n\nM.Sc\n\nin\n\nInformation Techn...,


In [13]:
candidate_documents_df.to_parquet(

    "/kaggle/working/candidate_documents.parquet",

    index=False

)

print("Saved Successfully")

Saved Successfully


In [14]:
import os

os.listdir("/kaggle/working")

['.virtual_documents', 'candidate_documents.parquet']

In [15]:
candidate_documents_df.shape

(100000, 6)

In [16]:
candidate_documents_df.iloc[0]

candidate_id                                                   CAND_0000001
profile_document          Backend Engineer | SQL, Spark, Cloud Software ...
career_document           \nThe candidate currently works\nas a\n\nBacke...
skills_document           \nThe candidate has\n\nintermediate\n\nprofici...
education_document        \nCompleted\n\nB.E.\n\nin\n\nComputer Science\...
certification_document                                                     
Name: 0, dtype: object

In [17]:
# ==========================================================
# Behavior Feature Engineering
# ==========================================================

from datetime import datetime
import pandas as pd
from tqdm.auto import tqdm

TODAY = pd.Timestamp("2026-06-01")   # Dataset reference date

behavior_rows = []

for candidate in tqdm(candidates, desc="Building Behavior Features"):

    s = candidate["redrob_signals"]

    # Days since last active
    last_active = pd.to_datetime(s["last_active_date"])
    days_inactive = (TODAY - last_active).days

    # Skill assessment
    assessments = s.get("skill_assessment_scores", {})
    avg_skill_score = (
        sum(assessments.values()) / len(assessments)
        if len(assessments) > 0 else 0
    )

    behavior_rows.append({

        "candidate_id": candidate["candidate_id"],

        # Availability
        "open_to_work": int(s["open_to_work_flag"]),
        "notice_period": s["notice_period_days"],
        "days_inactive": days_inactive,

        # Responsiveness
        "response_rate": s["recruiter_response_rate"],
        "response_time": s["avg_response_time_hours"],

        # Profile Quality
        "profile_complete": s["profile_completeness_score"],

        # Technical
        "github_score": s["github_activity_score"],
        "avg_skill_assessment": avg_skill_score,
        "endorsements_received": s["endorsements_received"],

        # Recruiter Interest
        "profile_views": s["profile_views_received_30d"],
        "search_appearance": s["search_appearance_30d"],
        "saved_by_recruiters": s["saved_by_recruiters_30d"],
        "interview_completion": s["interview_completion_rate"],
        "offer_acceptance": s["offer_acceptance_rate"],

        # Trust
        "verified_email": int(s["verified_email"]),
        "verified_phone": int(s["verified_phone"]),
        "linkedin_connected": int(s["linkedin_connected"])
    })

behavior_df = pd.DataFrame(behavior_rows)

print(behavior_df.shape)
behavior_df.head()

Building Behavior Features:   0%|          | 0/100000 [00:00<?, ?it/s]

(100000, 18)


,candidate_id,open_to_work,notice_period,days_inactive,response_rate,response_time,profile_complete,github_score,avg_skill_assessment,endorsements_received,profile_views,search_appearance,saved_by_recruiters,interview_completion,offer_acceptance,verified_email,verified_phone,linkedin_connected
0,CAND_0000001,1,60,12,0.34,177.8,86.9,9.2,49.725,35,23,249,4,0.71,0.58,1,1,0
1,CAND_0000002,1,60,201,0.29,171.6,78.7,-1.0,0.000,3,7,107,10,0.62,-1.00,0,0,0
2,CAND_0000003,0,150,72,0.46,119.4,31.9,-1.0,0.000,46,1,28,4,0.86,-1.00,1,0,0
3,CAND_0000004,0,120,68,0.26,104.1,28.5,-1.0,0.000,22,3,5,8,0.35,-1.00,1,1,1
4,CAND_0000005,1,30,243,0.37,116.7,84.6,-1.0,0.000,14,12,67,1,0.74,-1.00,0,1,1


In [18]:
behavior_df.describe().T

,count,mean,std,min,25%,50%,75%,max
open_to_work,100000.0,0.353390,0.478025,0.00,0.00,0.00,1.00,1.00
notice_period,100000.0,87.385800,36.589628,0.00,60.00,90.00,120.00,150.00
days_inactive,100000.0,113.723890,66.588459,5.00,57.00,110.00,167.00,245.00
response_rate,100000.0,0.436574,0.214122,0.02,0.25,0.44,0.62,0.95
response_time,100000.0,132.702744,75.238241,2.10,68.30,129.90,193.30,280.00
profile_complete,100000.0,56.758180,17.274069,25.00,42.20,56.80,71.60,99.90
github_score,100000.0,9.619230,17.761394,-1.00,-1.00,-1.00,16.70,96.90
avg_skill_assessment,100000.0,12.402740,23.062725,0.00,0.00,0.00,0.00,93.90
endorsements_received,100000.0,30.068570,20.242847,0.00,14.00,28.00,43.00,242.00
profile_views,100000.0,47.985250,32.051870,0.00,23.00,45.00,68.00,374.00


In [19]:
behavior_df.isnull().sum()

candidate_id             0
open_to_work             0
notice_period            0
days_inactive            0
response_rate            0
response_time            0
profile_complete         0
github_score             0
avg_skill_assessment     0
endorsements_received    0
profile_views            0
search_appearance        0
saved_by_recruiters      0
interview_completion     0
offer_acceptance         0
verified_email           0
verified_phone           0
linkedin_connected       0
dtype: int64

In [20]:
behavior_df.to_parquet(
    "/kaggle/working/behavior_features.parquet",
    index=False
)

print("Behavior Features Saved")

Behavior Features Saved


In [21]:
import os

print(os.listdir("/kaggle/working"))

['.virtual_documents', 'candidate_documents.parquet', 'behavior_features.parquet']


In [22]:
print(type(behavior_df))
print(behavior_df.shape)
behavior_df.head()

<class 'pandas.core.frame.DataFrame'>
(100000, 18)


,candidate_id,open_to_work,notice_period,days_inactive,response_rate,response_time,profile_complete,github_score,avg_skill_assessment,endorsements_received,profile_views,search_appearance,saved_by_recruiters,interview_completion,offer_acceptance,verified_email,verified_phone,linkedin_connected
0,CAND_0000001,1,60,12,0.34,177.8,86.9,9.2,49.725,35,23,249,4,0.71,0.58,1,1,0
1,CAND_0000002,1,60,201,0.29,171.6,78.7,-1.0,0.000,3,7,107,10,0.62,-1.00,0,0,0
2,CAND_0000003,0,150,72,0.46,119.4,31.9,-1.0,0.000,46,1,28,4,0.86,-1.00,1,0,0
3,CAND_0000004,0,120,68,0.26,104.1,28.5,-1.0,0.000,22,3,5,8,0.35,-1.00,1,1,1
4,CAND_0000005,1,30,243,0.37,116.7,84.6,-1.0,0.000,14,12,67,1,0.74,-1.00,0,1,1


In [23]:
behavior_df.to_parquet(
    "/kaggle/working/behavior_features.parquet",
    index=False
)

candidate_documents_df.to_parquet(
    "/kaggle/working/candidate_documents.parquet",
    index=False
)

In [40]:
# ==========================================================
# Production Candidate Preprocessing Pipeline
# ==========================================================

from tqdm.auto import tqdm
import numpy as np
import pandas as pd

documents = []
metadata = []
behavior = []
integrity = []

In [44]:
for candidate in tqdm(candidates, desc="Processing Candidates"):

    profile = candidate.get("profile", {})
    career = candidate.get("career_history", [])
    skills = candidate.get("skills", [])
    education = candidate.get("education", [])
    certs = candidate.get("certifications", [])
    signals = candidate.get("redrob_signals", {})

    #######################################################
    # Candidate ID
    #######################################################

    cid = candidate["candidate_id"]

    #######################################################
    # 1. Semantic Documents
    #######################################################

    builder = CandidateSemanticBuilder(candidate)

    profile_doc = builder.profile_document()
    career_doc = builder.career_document()
    skills_doc = builder.skills_document()
    education_doc = builder.education_document()
    cert_doc = builder.certification_document()

    documents.append({

        "candidate_id": cid,

        "profile_document": profile_doc,

        "career_document": career_doc,

        "skills_document": skills_doc,

        "education_document": education_doc,

        "certification_document": cert_doc

    })

    #######################################################
    # 2. Metadata
    #######################################################

    metadata.append({

        "candidate_id": cid,

        "headline": profile.get("headline"),

        "current_title": profile.get("current_title"),

        "current_company": profile.get("current_company"),

        "industry": profile.get("current_industry"),

        "country": profile.get("country"),

        "years_experience": profile.get("years_of_experience"),

        "highest_degree":
            education[-1]["degree"] if education else None,

        "num_jobs": len(career),

        "num_skills": len(skills),

        "num_certifications": len(certs)

    })

    #######################################################
    # 3. Behavior Features
    #######################################################

    behavior.append({

        "candidate_id": cid,

        "open_to_work":
            signals.get("open_to_work_flag", 0),

        "response_rate":
            signals.get("recruiter_response_rate", 0),

        "response_time":
            signals.get("avg_response_time_hours", 999),

        "profile_complete":
            signals.get("profile_completeness_score", 0),

        "github_score":
            signals.get("github_activity_score", -1),

        "endorsements":
            signals.get("endorsements_received", 0),

        "profile_views":
            signals.get("profile_views_received_30d", 0),

        "search_appearance":
            signals.get("search_appearance_30d", 0),

        "saved_by_recruiters":
            signals.get("saved_by_recruiters_30d", 0)

    })

    #######################################################
    # 4. Integrity Features
    #######################################################

    total_months = sum(
        j.get("duration_months", 0)
        for j in career
    )

    expected = (
        profile.get("years_of_experience", 0) * 12
    )

    gap = abs(total_months - expected)

    career_score = max(
        0,
        1 - gap / max(expected, 1)
    )

    expert_short = sum(

        1

        for s in skills

        if (
            str(s.get("proficiency", "")).lower() == "expert"
            and s.get("duration_months", 0) < 12
        )

    )

    education_issue = sum(

        1

        for e in education

        if e.get("end_year", 0) < e.get("start_year", 0)

    )

    integrity_score = (

        career_score

        - 0.10 * expert_short

        - 0.20 * education_issue

    )

    integrity_score = np.clip(
        integrity_score,
        0,
        1
    )

    integrity.append({

        "candidate_id": cid,

        "career_score": career_score,

        "expert_short": expert_short,

        "education_issue": education_issue,

        "integrity_score": integrity_score

    })

Processing Candidates:   0%|          | 0/100000 [00:00<?, ?it/s]

In [45]:
candidate_documents_df = pd.DataFrame(documents)

candidate_metadata_df = pd.DataFrame(metadata)

behavior_df = pd.DataFrame(behavior)

integrity_df = pd.DataFrame(integrity)

print(candidate_documents_df.shape)
print(candidate_metadata_df.shape)
print(behavior_df.shape)
print(integrity_df.shape)

(100000, 6)
(100000, 11)
(100000, 10)
(100000, 5)


In [46]:
candidate_documents_df.to_parquet(
    "/kaggle/working/candidate_documents.parquet",
    index=False
)

candidate_metadata_df.to_parquet(
    "/kaggle/working/candidate_metadata.parquet",
    index=False
)

behavior_df.to_parquet(
    "/kaggle/working/behavior_features.parquet",
    index=False
)

integrity_df.to_parquet(
    "/kaggle/working/integrity_features.parquet",
    index=False
)

print("All files saved successfully!")

All files saved successfully!
